# Deep Learning 012 — Multi-Class Classification with Keras: MNIST

Companion notebook to the lesson. Ten classes instead of two, and three things change:

1. the output layer becomes **10 nodes with softmax** instead of 1 with sigmoid,
2. the loss becomes **categorical cross-entropy**,
3. the 28×28 image has to be **flattened to 784 numbers** before a Dense layer will take it.

The third one is the interesting one, and this notebook spends most of its time on what it
costs.

> **TensorFlow is optional.** The parameter arithmetic and the softmax/normalisation
> experiments run on numpy and scikit-learn alone; scikit-learn's 8×8 `digits` dataset
> stands in for MNIST so every claim here produces a number. The Keras cells are marked.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## Part A — Where the parameters go

Architecture: `Flatten → Dense(128, relu) → Dense(10, softmax)`. Count it by hand before
asking a library.

In [ ]:
def dense_params(n_in, n_out):
    return n_in * n_out + n_out              # weights + one bias per output node

layers = [("Flatten 28x28 -> 784", 0),
          ("Dense(128)", dense_params(784, 128)),
          ("Dense(10)", dense_params(128, 10))]

total = sum(p for _, p in layers)
print(f"{'layer':>24}{'parameters':>13}{'share':>9}")
for name, p in layers:
    print(f"{name:>24}{p:>13,}{p / total:>9.1%}")
print(f"{'TOTAL':>24}{total:>13,}")
assert total == 101_770

**101,770 parameters, and 98.7% of them are in the first Dense layer.**

Two things follow.

`Flatten` has **zero** parameters — it is a reshape, not a layer that learns. But it is not
free, because of what it destroys.

In [ ]:
img = np.arange(784).reshape(28, 28)
flat = img.reshape(-1)
print("pixel (0,0) and its neighbour below, (1,0):")
print(f"  in the image : distance 1 row")
print(f"  after flatten: indices {np.where(flat == img[0, 0])[0][0]} and "
      f"{np.where(flat == img[1, 0])[0][0]} -- 28 apart")
print(f"\nand its neighbour to the right, (0,1): index "
      f"{np.where(flat == img[0, 1])[0][0]} -- 1 apart")
print("\nThe Dense layer sees 784 unordered numbers. Nothing in its weights knows that")
print("index 0 and index 28 were touching. That is what a CNN fixes, in lesson 037.")

In [ ]:
# the same architecture on a 28x28 image vs a modest 224x224 photo
for h, w, label in ((28, 28, "MNIST"), (224, 224, "a small colour photo (x3 channels)")):
    n_in = h * w * (3 if h == 224 else 1)
    p = dense_params(n_in, 128) + dense_params(128, 10)
    print(f"{label:<36} input {n_in:>7,}  ->  {p:>12,} parameters")

Nineteen million parameters for one small photo, before the network has learned anything at
all. Flattening does not scale, which is the entire motivation for convolution.

## Part B — Why softmax, not ten sigmoids

Ten sigmoid outputs would each answer "is this a 7?" independently, and could happily
report 0.9 for both 3 and 8. Softmax makes the ten outputs **compete**: they are forced to
sum to 1, so confidence in one class is confidence taken from the others.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)

logits = np.array([1.0, 0.4, 3.2, 0.1, -0.5, 0.8, 0.2, 2.9, 0.6, -1.1])
s, m = sigmoid(logits), softmax(logits)
print(f"{'class':>6}{'sigmoid':>10}{'softmax':>10}")
for i in range(10):
    print(f"{i:>6}{s[i]:>10.4f}{m[i]:>10.4f}")
print(f"{'sum':>6}{s.sum():>10.4f}{m.sum():>10.4f}")
print("\nthe sigmoid column is not a probability distribution; the softmax column is")

Note classes 2 and 7 in the softmax column — the model is genuinely torn between them, and
the numbers say so honestly because they had to share. Ten independent sigmoids can be
confident about both at once, which is not an opinion about a handwritten digit.

## Part C — Normalising the pixels

MNIST pixels are integers 0–255. Divide by 255 and every input lands in [0, 1]. No
`StandardScaler` is needed, because the range is known in advance rather than estimated
from the data — one of the rare cases where you can normalise without fitting anything.

That matters more than it looks. Un-normalised inputs make the weighted sums large, and a
large weighted sum saturates the activation.

In [ ]:
r = np.random.default_rng(1)
W = r.normal(scale=0.05, size=(784, 128))         # a typical initialisation
raw = r.integers(0, 256, size=(64, 784)).astype(float)

for name, X_ in (("raw 0-255", raw), ("divided by 255", raw / 255)):
    z = X_ @ W
    print(f"{name:>16}: |z| mean {np.abs(z).mean():>8.2f}   "
          f"sigmoid'(z) mean {(sigmoid(z) * (1 - sigmoid(z))).mean():.2e}")
print("\nthe raw inputs saturate the activation, and a saturated activation has no gradient")

## Part D — The whole pipeline, on data that ships with scikit-learn

`load_digits` is 8×8 rather than 28×28, and 1,797 images rather than 70,000, but every
structural decision is identical: flatten, normalise, hidden layer, 10-way softmax output.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

d = load_digits()
X, y = d.data, d.target
print(f"images {d.images.shape}, flattened {X.shape}, pixel range {X.min():.0f}-{X.max():.0f}")

Xn = X / 16.0                                    # this dataset's max is 16, not 255
X_tr, X_te, y_tr, y_te = train_test_split(Xn, y, test_size=0.2, random_state=0, stratify=y)

clf = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", solver="adam",
                    max_iter=300, random_state=0)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)
print(f"\ntest accuracy {accuracy_score(y_te, pred):.4f}   (10 classes, chance = 0.100)")
print(f"parameters: {dense_params(64, 128) + dense_params(128, 10):,}")

In [ ]:
# what it gets wrong, and it is never random
cm = confusion_matrix(y_te, pred)
print("   " + "".join(f"{i:>4}" for i in range(10)))
for i in range(10):
    print(f"{i:>2} " + "".join(f"{v:>4}" for v in cm[i]))

errs = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j and cm[i, j]]
print("\nmost common confusions:")
for n, i, j in sorted(errs, reverse=True)[:4]:
    print(f"  {n} x  true {i} predicted {j}")

Nine errors out of 360, so resist reading too much into the pattern — at this sample size
the difference between "2 x true 8 predicted 1" and any other pair is not significant. The
honest summary is that the diagonal holds essentially all the mass, which is what a working
10-way classifier looks like. Run the same confusion matrix on full MNIST in Part E, where
there are enough errors to count, and see whether the off-diagonal mass really does land on
visually similar pairs — do not take anyone's word for which pairs those are.

## Part E — The Keras version

Marked, because it needs TensorFlow. Everything above is what it is doing internally.

In [ ]:
# --- needs TensorFlow ---
import tensorflow as tf
from tensorflow import keras

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train.shape, x_test.shape, x_train.dtype, x_train.max())

x_train = x_train / 255.0                        # known range, no scaler needed
x_test = x_test / 255.0

model = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),  # 0 parameters, destroys the 2-D structure
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",   # integer labels, not one-hot
              metrics=["accuracy"])
model.summary()                                   # should print 101,770

history = model.fit(x_train, y_train, epochs=10, batch_size=32,
                    validation_split=0.2, verbose=1)
print(model.evaluate(x_test, y_test, verbose=0))

Note `sparse_categorical_crossentropy` rather than `categorical_crossentropy`. The labels
are integers 0–9, not one-hot vectors, and the sparse version indexes straight into the
softmax output instead of building a 10-wide one-hot array for every example. Same loss,
less memory — lesson 014 shows they are numerically identical.

## Overfitting, and what to do about it

With 101,770 parameters and 60,000 examples, overfitting is expected rather than
surprising. The signature is training loss still falling while validation loss turns
upward.

```python
import matplotlib.pyplot as plt
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.legend()
```

The standard responses, in the order worth trying them: **more data**, **dropout**, **L2
regularisation**, **early stopping**, **a smaller network**. Lessons 026–031 take these one
at a time.

## Try it yourself

1. Compute the parameter count for `Dense(512)` instead of `Dense(128)`. What fraction is
   still in the first layer?
2. Train the Part D model on un-normalised `X` and compare accuracy and iterations to
   converge.
3. Replace the softmax output with 10 independent sigmoids and a binary cross-entropy per
   class. Does accuracy drop? Do the outputs still sum to anything sensible?
4. In the confusion matrix, look up the images behind the most common confusion and decide
   whether *you* would get them right.